<a href="https://colab.research.google.com/github/joemarshall/websensors/blob/main/assets/python/KnockLock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This colab workbook presents a workflow for training a machine learning model for a simple pattern based recognition, then outputs it as a tflite file which can be used in the websensor platform or on a raspberry pi.

In [5]:

!pip install ai_edge_litert tensorflow pandas numpy
# tensorflow is the machine learning library we use
import tensorflow as tf
# numpy is for fast python maths
import numpy as np
# pandas for importing datafiles
import pandas as pd
import io

# make some stuff that is in tensorflow be
# easier to get at below
import tensorflow.keras as keras
import tensorflow.keras.layers as layers
import tensorflow.keras.losses as losses




In [6]:
# load datafiles - each datafile is a csv file of continuous sensor data.
# The accompanying ground truth is 1 during a correct unlock sequence, 2 during
# a failed unlock sequence, 0 otherwise

# recorded data can be from a raspberry pi with two buttons which are held down during
# the knock sequences, or from the websensor platform

# this stuff makes an upload box appear
from google.colab import files
uploaded = files.upload()

Saving knock_bad.csv to knock_bad (1).csv
Saving knock_good.csv to knock_good (1).csv


In [17]:
# preprocess - for each knock sequence take the time that the
# sequence takes and spread the raw data over 512 points
# and normalise to 0-1 to avoid differences in sensitivity between sensors
# or hardness of knock
# We also trim silence off the knock pattern
MIN_SAMPLES=50
column_names=["sound level","ground truth"]

AUGMENTED_DATA_POINTS=10

all_knocks=[]

def resample_knock(knock_index,sensor_data,gt_val,silence_start,silence_end):
  # normalize the value of sensor data
  sensor_np=np.array(sensor_data)
  # trim silence off the end
  last_sound_pos=np.argmax(np.flip(sensor_np)>128)
  sensor_np=sensor_np[:-last_sound_pos]
  sensor_np=sensor_np-np.min(sensor_np) # min is now zero
  max_sensor=np.max(sensor_np)
  if max_sensor>0:
    sensor_np=sensor_np/max_sensor # max is now one

  sensor_np = np.hstack([np.zeros((silence_start,)),sensor_np,np.zeros((silence_start,))])

  # resample it to 512 samples long
  x_out_positions=np.linspace(0,512,512)
  x_original=np.linspace(0,512,len(sensor_np))
  sensor_resampled=np.interp(x_out_positions,x_original,sensor_np)
  # make it have 2 axes - time point, sensor, because this is what tensorflow expects
  sensor_resampled=np.expand_dims(sensor_resampled,-1)
  # we want only zero and one, because we don't use any bits where no
  # button is pressed
  gt_val=gt_val-1
  return [knock_index,sensor_resampled,gt_val]

def preprocess_knock(knock_index,sensor_data,gt_val):
  knocks = [resample_knock(knock_index,sensor_data,gt_val,0,0)]
  silence_offsets = np.random.randint(0,15,size=(AUGMENTED_DATA_POINTS,2))
  for x in range(AUGMENTED_DATA_POINTS):
    knocks.append(resample_knock(knock_index,sensor_data,gt_val,silence_offsets[x,0],silence_offsets[x,1]))
  return knocks

knock_index=0
for c in uploaded.keys():
  print(f"Loading: {c}")
  csv_frame=pd.read_csv(io.BytesIO(uploaded[c]))
  last_gt=0
  press_buffer=None
  for i,row in csv_frame.iterrows():
    gt=row[column_names[1]]
    sensor=row[column_names[0]]
    if gt!=last_gt:
      if gt==0:
        # finished pressing - add this to our list of knocks
        # as long as it is more than 50 samples
        # otherwise assume error
        if len(press_buffer)>MIN_SAMPLES:
          # add this knock to our data
          all_knocks.extend(preprocess_knock(knock_index,press_buffer,last_gt))
          knock_index+=1
      else:
        # starting a press, clear the press buffer
        press_buffer=[]
    if gt!=0:
      press_buffer.append(sensor)
    last_gt=gt





Loading: knock_bad (1).csv
Loading: knock_good (1).csv


In [27]:
# shuffle the knocks in the dataset
# (we want all augmentations of each knock to be in
# either training or test, and given we augmented on import
# we can't just shuffle everything
p = np.random.permutation(knock_index)
test_cutoff= (len(p)*3)//4
training_p = p[:test_cutoff]
test_p = p[test_cutoff:]
print([i for (i,x,y) in all_knocks])
print(p,training_p)
x_train = np.stack([x for (i,x,y) in all_knocks if i in training_p])
y_train = np.stack([y for (i,x,y) in all_knocks if i in training_p])

x_test = np.stack([x for (i,x,y) in all_knocks if i in test_p])
y_test = np.stack([y for (i,x,y) in all_knocks if i in test_p])

print(f"Loaded data: Train:{x_train.shape},{y_train.shape}")
print(f"Loaded data: Test:{x_test.shape},{y_test.shape}")



[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18, 18, 18, 18, 18, 18, 18, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 25, 25, 2

In [31]:
# build a model - 4 convolutional layers to identify features, then a fully connected layer to output
# the unlock or not unlock inference
model=keras.Sequential(layers=[layers.Input(name='x',shape=(512,1),batch_size=1),layers.Conv1D(32,kernel_size=3,padding="same",strides=2,activation="relu"),
                         layers.Conv1D(32,kernel_size=3,padding="same",strides=2,activation="relu"),
                         layers.Conv1D(32,kernel_size=3,padding="same",strides=2,activation="relu"),
                         layers.Conv1D(32,kernel_size=3,padding="same",strides=2,activation="relu"),
                         #layers.Conv1D(64,kernel_size=32,padding="same",strides=32,activation="relu"),
                         layers.Flatten(),
                         layers.Dense(64,activation="relu"),
                         layers.Dense(2,activation="softmax",name='y')])
model.compile(optimizer='adam', loss=losses.SparseCategoricalCrossentropy(),metrics=['accuracy'])
model.build(input_shape=(1,512,1))
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_8 (Conv1D)               │ (1, 256, 32)           │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (1, 128, 32)           │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (1, 64, 32)            │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (1, 32, 32)            │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (1, 1024)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (1, 64)                │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y (Dense)                       │ (1, 2)                 │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,170 (293.63 KB)

 Trainable params: 75,170 (293.63 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
# call train on the model
model.fit(x_train,y_train,batch_size=1,validation_data=(x_test,y_test),epochs=100)


Epoch 1/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6132 - loss: 0.6413 - val_accuracy: 0.6883 - val_loss: 0.7274
Epoch 2/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9605 - loss: 0.2383 - val_accuracy: 0.7143 - val_loss: 1.1505
Epoch 3/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9791 - loss: 0.0711 - val_accuracy: 0.7273 - val_loss: 1.6861
Epoch 4/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9765 - loss: 0.0783 - val_accuracy: 0.7143 - val_loss: 2.7785
Epoch 5/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 0.7143 - val_loss: 3.3541
Epoch 6/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 7.4032e-04 - val_accuracy: 0.7143 - val_loss: 3.7121
Epoch 7/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 1.0000 - loss: 2.2020e-04 - val_accuracy: 0.7143 - val_loss: 3.7811
Epoch 8/100
220/220 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 1.0000 - loss: 1.2693e-0

In [33]:
#plot confusion matrix so we can see how it predicts for different categories
test_prediction=tf.math.argmax(model.predict(x_test),axis=-1)
train_prediction=tf.math.argmax(model.predict(x_train),axis=-1)
print("Test predictions")
print(tf.math.confusion_matrix(y_test,test_prediction))
print("train predictions")
print(tf.math.confusion_matrix(y_train,train_prediction))

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Test predictions
tf.Tensor(
[[44 11]
 [11 11]], shape=(2, 2), dtype=int32)
train predictions
tf.Tensor(
[[ 77   0]
 [  0 143]], shape=(2, 2), dtype=int32)


In [35]:
# Save model to a tflite model for inference on raspberry pi (or websensor platform)
converter=tf.lite.TFLiteConverter.from_keras_model(model)
tflite=converter.convert()

tflite_model_file = open('model.tflite',"wb")
tflite_model_file.write(tflite)


from ai_edge_litert.compiled_model import CompiledModel

lite_model = CompiledModel.from_file("model.tflite")
print(lite_model,lite_model.get_signature_by_index(0))
print(dir(lite_model))
from google.colab import files
files.download('model.tflite')


Saved artifact at '/tmp/tmpmlw_qv4l'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 512, 1), dtype=tf.float32, name='x')
Output Type:
  TensorSpec(shape=(1, 2), dtype=tf.float32, name=None)
Captures:
  136081411315600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411322512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411322320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411320016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411323088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411322704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411323472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411323280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411323856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411323664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136081411324240: TensorSpec(shape=()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>